homo: std=0.6022 eV | mean-predictor MAE=0.4433 eV  
lumo: std=1.2772 eV | mean-predictor MAE=1.0518 eV

The LUMO pred in first model was less acurate because of bigger spread

Record Ridge result

In [1]:
from qm9lib.data import save_params
save_params("cfmf_ridge", {
    "features": {"min_count": 1, "max_radius": 2, "binary": False},
    "homo": {"alpha": 1.0},
    "lumo": {"alpha": 0.3},
})

=== homo: top 5 ===
 subsample  reg_lambda  num_leaves  min_child_samples  colsample_bytree  n_trees  val_mae        sec
      0.85           5         255                 10               1.0     4948 0.100466 374.275702
      1.00           5         255                 10               1.0     4041 0.100577 266.796360
      1.00           1         255                 10               0.7     5000 0.100771 316.815419
      0.85           1         127                  5               1.0     4988 0.103377 196.164057
      0.85           0         127                 10               0.7     4199 0.103941 156.833490

=== lumo: top 5 ===
 subsample  reg_lambda  num_leaves  min_child_samples  colsample_bytree  n_trees  val_mae        sec
      1.00           1         255                 10               0.7     5000 0.098334 327.584699
      0.85           5         255                 10               1.0     5000 0.098842 364.643152
      1.00           5         255                 10               1.0     5000 0.100424 348.917241
      0.85           0         127                 10               0.7     5000 0.101773 186.855308
      0.85           1         127                  5               1.0     5000 0.102200 194.208537

saved -> best_params.json['cfmf_lgbm']
reference val MAE (eV): Ridge CF-MF 0.120 / 0.144 | RF descriptors 0.121 / 0.128

In [ ]:
import time
import joblib
from qm9lib.cfmf import load_cfmf

booster = joblib.load("models/cfmf_lgbm_random_homo.joblib")
(_, _, X_te), _, _ = load_cfmf("random", min_count=10, max_radius=2, binary=False)

print(f"trees: {booster.num_trees()}")

for n in (50, 200):
    t0 = time.time()
    booster.predict(X_te[:n], pred_contrib=True)
    dt = time.time() - t0
    print(f"{n:>4} molecules: {dt:.1f}s  ({dt / n * 1000:.0f} ms/molecule)")

trees: 11408
  50 molecules: 533.5s  (10670 ms/molecule)


In [2]:
import joblib
booster = joblib.load("models/cfmf_lgbm_random_homo.joblib")
print(booster.params.get("num_threads"), booster.params.get("nthread"))

None None


In [ ]:
# scratch: accuracy and SHAP cost vs tree count
import time

import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error

from qm9lib.cfmf import load_cfmf

TARGET = "homo"
N_TREES = [500, 1000, 2000, 4000, 8000]
N_BENCH = 50

booster = joblib.load(f"models/cfmf_lgbm_random_{TARGET}.joblib")
(_, _, X_te), y, _ = load_cfmf("random", min_count=10, max_radius=2, binary=False)
y_te = y[TARGET][2]
full = booster.num_trees()
print(f"full model: {full} trees\n")

print(f"{'trees':>6} {'test MAE':>9} {'ms/mol':>8} {'3000 mol':>10}")
for n in N_TREES:
    k = full if n is None else n
    mae = mean_absolute_error(y_te, booster.predict(X_te, num_iteration=k))
    t0 = time.time()
    booster.predict(X_te[:N_BENCH], pred_contrib=True, num_iteration=k)
    ms = (time.time() - t0) / N_BENCH * 1000
    print(f"{k:>6} {mae:>9.4f} {ms:>8.0f} {ms * 3000 / 60000:>9.1f}m")

full model: 11408 trees

 trees  test MAE   ms/mol   3000 mol
   500    0.1183      431      21.5m
  1000    0.1091      778      38.9m
  2000    0.1037     1422      71.1m
  4000    0.1005     3140     157.0m


In [1]:
# scratch: smaller LightGBM for interpretation + SHAP cost check
import time

import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error

from config import MODEL_DIR, TARGETS
from qm9lib.cfmf import load_cfmf
from qm9lib.models import fit_lgbm

PARAMS = {
    "num_leaves": 63,
    "min_child_samples": 20,
    "colsample_bytree": 0.7,
    "subsample": 0.85,
    "reg_lambda": 1,
}

(X_tr, X_va, X_te), y, _ = load_cfmf("random", min_count=10, max_radius=2, binary=False)

for t in TARGETS:
    y_tr, y_va, y_te = y[t]
    t0 = time.time()
    b = fit_lgbm(X_tr, y_tr, X_va, y_va, PARAMS,
                 learning_rate=0.1, max_rounds=6000, patience=100, log_period=2000)
    mae = mean_absolute_error(y_te, b.predict(X_te))
    joblib.dump(b, MODEL_DIR / f"cfmf_lgbm_small_random_{t}.joblib")

    t1 = time.time()
    b.predict(X_te[:50], pred_contrib=True)
    ms = (time.time() - t1) / 50 * 1000
    print(f"{t}: {b.num_trees()} trees  test MAE={mae:.4f}  "
          f"fit={time.time()-t0:.0f}s  SHAP={ms:.0f} ms/mol  "
          f"-> 2000 mol = {ms*2000/60000:.1f} min")

[2000]	valid_0's l1: 0.114968
[4000]	valid_0's l1: 0.110283
[6000]	valid_0's l1: 0.108297
homo: 6000 trees  test MAE=0.1071  fit=116s  SHAP=108 ms/mol  -> 2000 mol = 3.6 min
[2000]	valid_0's l1: 0.116865
[4000]	valid_0's l1: 0.110073
[6000]	valid_0's l1: 0.107166
lumo: 5999 trees  test MAE=0.1054  fit=119s  SHAP=100 ms/mol  -> 2000 mol = 3.3 min


In [2]:
import pandas as pd
tab = pd.read_parquet("data/shap_ridge_lumo.parquet")
carbonyl = tab[(tab.radius == 1) & (tab.n_molecules >= 200)
               & tab.fragment.str.contains("=", na=False)]
print(carbonyl[["fragment", "weight", "n_molecules", "example_smiles"]]
      .sort_values("weight").to_string(index=False, float_format="%.4f"))

            fragment  weight  n_molecules   example_smiles
            [NH]=[C] -0.3730          630 C#CC#CC1CC(=N)O1
            [O]=[CH] -0.2213         1739     C#CC#CC#CC=O
     [CH](=[C])[CH2] -0.0859          278   C#CC#CC1=CCCC1
[C](=[O])([CH])[CH2] -0.0830          272   C#CC#CC1CCC1=O
      [CH](=[CH])[C] -0.0650          298 C#CC1(C#C)C=CCC1
             [O]=[C] -0.0314         2273 C#CC#CC1CC(=O)C1
     [CH]([CH2])=[O]  0.0100          293    C#CC#CC#CCC=O
     [CH](=[CH])[CH]  0.0247          602   C#CC#CC1C=CCC1
       [CH]([C])=[O]  0.0268          307 C#CC#CC1(C=O)CC1
    [CH](=[CH])[CH2]  0.1018          527   C#CC#CC1C=CCC1
      [CH]([CH])=[O]  0.1474          311   C#CC#CC1CC1C=O
             [O]=[C]  0.1557         1193  C#CC#CC#CC(C)=O


In [3]:
# scratch: check the assignment, especially aziridine and unassigned
import pandas as pd

from config import DATA_DIR

frags = pd.read_parquet(DATA_DIR / "cfmf_fragments.parquet")
groups = pd.read_parquet(DATA_DIR / "cfmf_groups.parquet")
shap = pd.read_parquet(DATA_DIR / "shap_ridge_lumo.parquet")
m = frags.join(groups.set_index("col")[["group", "all_groups"]], on="col") \
         .join(shap.set_index("col")[["weight", "mean_abs_shap", "n_molecules"]], on="col")

# 1. how many DISTINCT example molecules back each group?
print("=== distinct example molecules per group ===")
print(m.groupby("group")["example_mol_id"].nunique().sort_values().to_string())

# 2. what is aziridine actually matching?
print("\n=== aziridine: 10 columns with most support ===")
az = m[m.group == "aziridine"].nlargest(10, "n_molecules")
print(az[["radius", "fragment", "center_symbol", "n_molecules",
          "example_smiles"]].to_string(index=False))

# 3. what is in unassigned that matters?
print("\n=== unassigned: 15 columns with highest mean|SHAP| ===")
un = m[m.group == "unassigned"].nlargest(15, "mean_abs_shap")
print(un[["radius", "fragment", "center_symbol", "weight", "mean_abs_shap",
          "n_molecules"]].to_string(index=False, float_format="%.4f"))

# 4. which columns carry the importance at all? support distribution
print("\n=== importance by support band ===")
bands = pd.cut(m.n_molecules, [0, 10, 100, 1000, 20000],
               labels=["1-10", "11-100", "101-1000", ">1000"])
print(m.groupby(bands, observed=True)
       .agg(n_cols=("col", "count"), importance=("mean_abs_shap", "sum"))
       .assign(share=lambda d: d.importance / d.importance.sum())
       .to_string(float_format="%.4f"))

=== distinct example molecules per group ===
group
fluoride               2
aldehyde             469
nitrile              508
pyrrole-type NH      920
phenol OH            926
furan-type O        1007
primary amine       1182
alkyne              1223
ester               1725
pyridine-type N     2521
epoxide             3545
imine               3628
alkene              3857
amide               3929
secondary amine     6338
tertiary amine      7125
aziridine           7742
ketone              8199
alcohol OH          8771
ether O            15841
unassigned         46045

=== aziridine: 10 columns with most support ===
 radius                    fragment center_symbol  n_molecules   example_smiles
      0 [N; deg 2; 1H; chg 0; ring]             N         2564   C#CC#CC#CC1CN1
      1              [NH]([CH])[CH]             N          400  C#CC#CC1CC2NC12
      1        [CH]([CH])([CH2])[N]             C          381  C#CC#CC1C2CCN12
      1               [CH2]([C])[N]             C      

In [4]:
import pandas as pd
from config import DATA_DIR

f = pd.read_parquet(DATA_DIR / "cfmf_fragments.parquet")
g = pd.read_parquet(DATA_DIR / "cfmf_groups.parquet")
s = pd.read_parquet(DATA_DIR / "shap_ridge_homo.parquet")
m = f.join(g.set_index("col")[["group", "all_groups"]], on="col") \
     .join(s.set_index("col")[["weight", "mean_abs_shap", "n_molecules"]], on="col")

for grp in ("aldehyde", "ketone"):
    print(f"\n=== {grp}: columns with support >= 200 ===")
    print(m[(m.group == grp) & (m.n_molecules >= 200)]
          .nlargest(12, "mean_abs_shap")[
              ["radius", "fragment", "center_symbol", "weight",
               "n_molecules", "all_groups"]]
          .to_string(index=False, float_format="%.4f"))


=== aldehyde: columns with support >= 200 ===
 radius              fragment center_symbol  weight  n_molecules               all_groups
      0 [C; deg 2; 1H; chg 0]             C  0.2715         1974 aldehyde|carbon skeleton
      1              [O]=[CH]             O -0.0622         1739                 aldehyde
      1        [CH]([CH])=[O]             C  0.1207          311 aldehyde|carbon skeleton
      1         [CH]([C])=[O]             C  0.0432          307 aldehyde|carbon skeleton
      1       [CH]([CH2])=[O]             C  0.0432          293 aldehyde|carbon skeleton

=== ketone: columns with support >= 200 ===
 radius              fragment center_symbol  weight  n_molecules             all_groups
      0 [O; deg 1; 0H; chg 0]             O -0.4655         4890                 ketone
      0 [C; deg 3; 0H; chg 0]             C  0.3175         1373 carbon skeleton|ketone
      1               [O]=[C]             O -0.1292         2273                 ketone
      1         

In [5]:
# scratch: is the carbonyl's HOMO contribution context-dependent?
import pandas as pd
from rdkit import Chem

from config import DATA_DIR
from qm9lib.data import load_clean, load_splits
from qm9lib.explain import CARBONYL_MERGE, merge_groups

totals = pd.read_parquet(DATA_DIR / "group_shap_homo.parquet")
present = totals.notna()          # placeholder; rebuild presence from SMILES below
tm, _ = merge_groups(totals, present, CARBONYL_MERGE)

df = load_clean()
_, _, test_ids = load_splits("random")
smi = df.loc[test_ids, "canonical_smiles"]

P = {
    "carbonyl": Chem.MolFromSmarts("[CX3]=[OX1]"),
    "amine":    Chem.MolFromSmarts("[NX3;H2,H1,H0;!$(N[CX3]=[OX1])][#6]"),
    "alkene":   Chem.MolFromSmarts("[CX3]=[CX3]"),
    "alcohol":  Chem.MolFromSmarts("[OX2H1]"),
}
has = pd.DataFrame(
    {k: [Chem.MolFromSmiles(s).HasSubstructMatch(p) for s in smi] for k, p in P.items()},
    index=test_ids,
)

c = tm.loc[has["carbonyl"], "carbonyl C=O"]
h = has[has["carbonyl"]]

print("carbonyl HOMO contribution (eV), split by what else the molecule has\n")
for label, mask in [
    ("no amine, no alkene, no OH", ~h.amine & ~h.alkene & ~h.alcohol),
    ("has alcohol OH only",         ~h.amine & ~h.alkene &  h.alcohol),
    ("has alkene",                  ~h.amine &  h.alkene),
    ("has amine",                    h.amine),
]:
    v = c[mask.to_numpy()]
    if len(v) >= 30:
        print(f"{label:<28} n={len(v):>5}  median={v.median():+.3f}  "
              f"frac_neg={(v < 0).mean():.2f}")

carbonyl HOMO contribution (eV), split by what else the molecule has

no amine, no alkene, no OH   n= 2147  median=-0.022  frac_neg=0.56
has alcohol OH only          n=  820  median=-0.009  frac_neg=0.52
has alkene                   n=  421  median=-0.053  frac_neg=0.62
has amine                    n= 1191  median=-0.043  frac_neg=0.60


In [1]:
import pandas as pd

from config import DATA_DIR, TARGETS

FAMILIES = {
    "HallKierAlpha": "saturation / shape",
    "FractionCSP3": "saturation / shape",
    "NumAliphatic": "saturation / shape",
    "NumSaturated": "saturation / shape",
    "NumAromatic": "saturation / shape",
    "PEOE_VSA": "charge-weighted surface (PEOE)",
    "EState_VSA": "electrotopological surface",
    "VSA_EState": "electrotopological surface",
    "SMR_VSA": "refractivity-weighted surface",
    "SlogP_VSA": "logP-weighted surface",
    "BCUT2D": "BCUT eigenvalue",
    "fr_": "functional group count",
    "Chi": "connectivity index",
    "Kappa": "shape index",
}


def family_of(name):
    for key, label in FAMILIES.items():
        if name.startswith(key) or key in name:
            return label
    return "other"


for t in TARGETS:
    s = pd.read_csv(DATA_DIR / f"shap_desc_summary_{t}.csv", index_col=0)
    s["family"] = [family_of(c) for c in s.index]
    s.to_csv(DATA_DIR / f"shap_desc_summary_{t}.csv")

    fam = (s.groupby("family")["mean_abs_shap"]
             .agg(["sum", "count"]).sort_values("sum", ascending=False))
    fam["share"] = fam["sum"] / fam["sum"].sum()
    print(f"\n=== {t}: importance by descriptor family ===")
    print(fam.to_string(float_format="%.4f"))


=== homo: importance by descriptor family ===
                                  sum  count  share
family                                             
functional group count         0.1969     53 0.1926
other                          0.1677     31 0.1641
refractivity-weighted surface  0.1529      9 0.1496
charge-weighted surface (PEOE) 0.1452     14 0.1420
electrotopological surface     0.1449     19 0.1418
logP-weighted surface          0.0901      9 0.0882
BCUT eigenvalue                0.0604      8 0.0591
connectivity index             0.0309      6 0.0302
saturation / shape             0.0292     10 0.0286
shape index                    0.0039      3 0.0038

=== lumo: importance by descriptor family ===
                                  sum  count  share
family                                             
saturation / shape             0.6946     10 0.3758
other                          0.4008     31 0.2169
electrotopological surface     0.2842     19 0.1538
functional group count